# 🇻🇳 OCR Tiếng Việt với Tesseract - State-of-the-Art

## Mục tiêu
- Nhận dạng chữ tiếng Việt (có đầy đủ dấu) bằng Tesseract
- Áp dụng các thuật toán tiền xử lý ảnh tốt nhất
- Đạt độ chính xác cao với chữ tiếng Việt và các dấu thanh

## Các kỹ thuật sử dụng
1. **Tiền xử lý ảnh (Preprocessing)**: Denoise, Deskew, Binarization
2. **Super Resolution**: Tăng độ phân giải ảnh
3. **Tesseract với data tiếng Việt (`vie`)**: `--oem 3 --psm 6`
4. **Postprocessing**: Làm sạch kết quả
5. **Đánh giá chất lượng**: CER, WER

## 1. Cài đặt thư viện

In [4]:
# Cài đặt Tesseract và gói tiếng Việt
!apt-get install -y tesseract-ocr tesseract-ocr-vie
!pip install pytesseract opencv-python-headless Pillow numpy matplotlib scikit-image jiwer

'apt-get' is not recognized as an internal or external command,
operable program or batch file.


In [5]:
import cv2
import numpy as np
import pytesseract
from PIL import Image, ImageEnhance, ImageFilter
import matplotlib.pyplot as plt
from skimage import filters, morphology, exposure
from skimage.restoration import denoise_nl_means, estimate_sigma
import warnings
warnings.filterwarnings('ignore')

print('✅ Import thư viện thành công!')
print(f'Tesseract version: {pytesseract.get_tesseract_version()}')
langs = pytesseract.get_languages(config='')
print(f'Ngôn ngữ hỗ trợ: {langs}')
if 'vie' in langs:
    print('✅ Đã cài tiếng Việt (vie)!')
else:
    print('❌ Chưa có gói tiếng Việt, chạy lại ô cài đặt!')

✅ Import thư viện thành công!
Tesseract version: 5.5.0.20241111
Ngôn ngữ hỗ trợ: ['afr', 'amh', 'ara', 'asm', 'aze', 'aze_cyrl', 'bel', 'ben', 'bod', 'bos', 'bre', 'bul', 'cat', 'ceb', 'ces', 'chi_sim', 'chi_sim_vert', 'chi_tra', 'chi_tra_vert', 'chr', 'cos', 'cym', 'dan', 'deu', 'deu_latf', 'div', 'dzo', 'ell', 'eng', 'enm', 'epo', 'equ', 'est', 'eus', 'fao', 'fas', 'fil', 'fin', 'fra', 'frm', 'fry', 'gla', 'gle', 'glg', 'grc', 'guj', 'hat', 'heb', 'hin', 'hrv', 'hun', 'hye', 'iku', 'ind', 'isl', 'ita', 'ita_old', 'jav', 'jpn', 'jpn_vert', 'kan', 'kat', 'kat_old', 'kaz', 'khm', 'kir', 'kmr', 'kor', 'lao', 'lat', 'lav', 'lit', 'ltz', 'mal', 'mar', 'mkd', 'mlt', 'mon', 'mri', 'msa', 'mya', 'nep', 'nld', 'nor', 'oci', 'ori', 'osd', 'pan', 'pol', 'por', 'pus', 'que', 'ron', 'rus', 'san', 'sin', 'slk', 'slv', 'snd', 'spa', 'spa_old', 'sqi', 'srp', 'srp_latn', 'sun', 'swa', 'swe', 'syr', 'tam', 'tat', 'tel', 'tgk', 'tha', 'tir', 'ton', 'tur', 'uig', 'ukr', 'urd', 'uzb', 'uzb_cyrl', 'vie', '

## 2. Tạo ảnh mẫu tiếng Việt để test

In [6]:
from PIL import Image, ImageDraw, ImageFont
import urllib.request, os

# Tải font hỗ trợ Unicode (tiếng Việt)
FONT_URL = 'https://github.com/google/fonts/raw/main/ofl/notosans/NotoSans-Regular.ttf'
FONT_PATH = '/tmp/NotoSans-Regular.ttf'
if not os.path.exists(FONT_PATH):
    urllib.request.urlretrieve(FONT_URL, FONT_PATH)
    print('✅ Đã tải font NotoSans!')

# Văn bản tiếng Việt mẫu
SAMPLE_TEXT = (
    'Xin chào! Đây là bài kiểm tra OCR tiếng Việt.\n'
    'Việt Nam – đất nước hình chữ S xinh đẹp.\n'
    'Bầu trời xanh, biển cả mênh mông.\n'
    'Ngôn ngữ: à á â ã ä ả ạ ă ắ ặ ầ ề ộ ượ\n'
    'Số: 1234567890 – Ký tự: !@#$%^&*()'
)

def create_sample_image(text, font_size=36, bg_color=255, txt_color=0, noise_level=5):
    font = ImageFont.truetype(FONT_PATH, font_size)
    lines = text.split('\n')
    # Tính kích thước
    dummy = Image.new('L', (1, 1))
    d = ImageDraw.Draw(dummy)
    line_heights = [d.textbbox((0, 0), l, font=font)[3] for l in lines]
    max_w = max(d.textbbox((0, 0), l, font=font)[2] for l in lines)
    total_h = sum(line_heights) + 20 * len(lines)
    img = Image.new('L', (max_w + 60, total_h + 40), color=bg_color)
    draw = ImageDraw.Draw(img)
    y = 20
    for i, line in enumerate(lines):
        draw.text((30, y), line, fill=txt_color, font=font)
        y += line_heights[i] + 20
    # Thêm nhiễu Gaussian nhẹ
    arr = np.array(img, dtype=np.float32)
    arr += np.random.normal(0, noise_level, arr.shape)
    arr = np.clip(arr, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

sample_img = create_sample_image(SAMPLE_TEXT)
sample_img.save('/tmp/sample_viet.png')

plt.figure(figsize=(12, 5))
plt.imshow(sample_img, cmap='gray')
plt.title('Ảnh mẫu tiếng Việt', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()
print('✅ Đã tạo ảnh mẫu!')

HTTPError: HTTP Error 404: Not Found

## 3. Pipeline tiền xử lý ảnh (State-of-the-Art)

In [ ]:
class VietnameseOCRPreprocessor:
    """
    Pipeline tiền xử lý ảnh tối ưu cho OCR tiếng Việt.
    Áp dụng các thuật toán state-of-the-art:
    - Chuyển grayscale
    - Tăng độ phân giải (Super Resolution nhẹ)
    - Khử nhiễu (Non-local Means / Gaussian)
    - Deskew (chỉnh nghiêng)
    - Contrast Enhancement (CLAHE)
    - Binarization thích nghi (Sauvola)
    - Morphological cleanup
    """

    def to_grayscale(self, img_bgr):
        """Chuyển ảnh sang grayscale nếu cần."""
        if len(img_bgr.shape) == 3:
            return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        return img_bgr

    def upscale(self, gray, scale=2):
        """
        Tăng độ phân giải ảnh (giúp Tesseract nhận diện dấu tiếng Việt tốt hơn).
        Dùng INTER_CUBIC cho chất lượng cao.
        Chỉ upscale nếu ảnh quá nhỏ (<= 1000px chiều rộng).
        """
        h, w = gray.shape
        if w <= 1000:
            gray = cv2.resize(gray, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)
        return gray

    def denoise(self, gray):
        """
        Khử nhiễu:
        - Nếu ảnh nhiều nhiễu: dùng Non-local Means (chất lượng cao)
        - Mặc định: GaussianBlur nhẹ
        """
        # Ước lượng nhiễu
        sigma_est = np.mean(estimate_sigma(gray, channel_axis=None))
        if sigma_est > 15:
            # Nhiều nhiễu: dùng Non-local Means
            denoised = denoise_nl_means(gray, h=1.15 * sigma_est, fast_mode=True,
                                        patch_size=5, patch_distance=3)
            gray = (denoised * 255).astype(np.uint8)
        else:
            # Ít nhiễu: Gaussian nhẹ
            gray = cv2.GaussianBlur(gray, (3, 3), 0)
        return gray

    def deskew(self, gray):
        """
        Chỉnh nghiêng ảnh dùng Hough Transform.
        Phát hiện góc nghiêng và xoay lại.
        """
        # Nhị phân hoá tạm thời để phát hiện cạnh
        _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        coords = np.column_stack(np.where(bw > 0))
        if len(coords) < 100:
            return gray
        angle = cv2.minAreaRect(coords)[-1]
        if angle < -45:
            angle = -(90 + angle)
        else:
            angle = -angle
        (h, w) = gray.shape
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        rotated = cv2.warpAffine(gray, M, (w, h),
                                  flags=cv2.INTER_CUBIC,
                                  borderMode=cv2.BORDER_REPLICATE)
        return rotated

    def enhance_contrast(self, gray):
        """
        Tăng cường độ tương phản dùng CLAHE.
        Hiệu quả cho ảnh chụp thực tế, ánh sáng không đều.
        """
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        return clahe.apply(gray)

    def binarize(self, gray, method='sauvola'):
        """
        Nhị phân hoá (binarization) thích nghi:
        - 'sauvola': Sauvola (tốt nhất cho tài liệu, giữ dấu tiếng Việt)
        - 'otsu': Otsu global
        - 'adaptive': OpenCV adaptive threshold
        """
        if method == 'sauvola':
            thresh = filters.threshold_sauvola(gray, window_size=25, k=0.2)
            binary = (gray > thresh).astype(np.uint8) * 255
        elif method == 'otsu':
            _, binary = cv2.threshold(gray, 0, 255,
                                       cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        else:
            binary = cv2.adaptiveThreshold(
                gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                cv2.THRESH_BINARY, 21, 10
            )
        return binary

    def morphological_cleanup(self, binary):
        """
        Lọc morphological: loại noise nhỏ, nối nét đứt.
        Quan trọng cho dấu tiếng Việt (những nét nhỏ).
        """
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
        # Closing: lấp các khoảng trống nhỏ trong ký tự
        closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
        # Opening: loại noise nhỏ
        opened = cv2.morphologyEx(closed, cv2.MORPH_OPEN, kernel)
        return opened

    def preprocess(self, image_path, binarize_method='sauvola',
                   do_upscale=True, do_deskew=True, do_denoise=True,
                   do_contrast=True, do_morph=True, show=True):
        """
        Pipeline đầy đủ: đọc ảnh → xử lý → trả về ảnh đã xử lý.
        """
        img = cv2.imread(image_path)
        if img is None:
            raise FileNotFoundError(f'Không tìm thấy ảnh: {image_path}')

        steps = {'Original': cv2.cvtColor(img, cv2.COLOR_BGR2RGB)}

        gray = self.to_grayscale(img)
        steps['Grayscale'] = gray

        if do_upscale:
            gray = self.upscale(gray)
            steps['Upscale x2'] = gray

        if do_denoise:
            gray = self.denoise(gray)
            steps['Denoise'] = gray

        if do_deskew:
            gray = self.deskew(gray)
            steps['Deskew'] = gray

        if do_contrast:
            gray = self.enhance_contrast(gray)
            steps['CLAHE'] = gray

        binary = self.binarize(gray, method=binarize_method)
        steps[f'Binarize({binarize_method})'] = binary

        if do_morph:
            binary = self.morphological_cleanup(binary)
            steps['Morph Cleanup'] = binary

        if show:
            n = len(steps)
            fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
            if n == 1:
                axes = [axes]
            for ax, (title, image) in zip(axes, steps.items()):
                cmap = 'gray' if len(image.shape) == 2 else None
                ax.imshow(image, cmap=cmap)
                ax.set_title(title, fontsize=10)
                ax.axis('off')
            plt.suptitle('Pipeline Tiền xử lý ảnh', fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.show()

        return binary

print('✅ Đã định nghĩa VietnameseOCRPreprocessor!')

## 4. OCR với Tesseract - Cấu hình tối ưu cho tiếng Việt

In [ ]:
class VietnameseTesseractOCR:
    """
    Wrapper Tesseract OCR tối ưu cho tiếng Việt.
    
    Tham số Tesseract quan trọng:
    - --oem 3: LSTM + Legacy (tốt nhất, kết hợp cả 2 engine)
    - --psm 6: Giả sử khối văn bản đều (phổ biến nhất)
    - lang='vie': Dữ liệu huấn luyện tiếng Việt
    - lang='vie+eng': Kết hợp Việt + Anh (nếu có số, ký tự La-tinh)
    """

    # Các chế độ PSM của Tesseract
    PSM_MODES = {
        0: 'OSD only (phát hiện hướng/chữ)',
        1: 'Auto OSD',
        2: 'Auto (không OSD)',
        3: 'Auto đầy đủ (default)',
        4: 'Một cột văn bản (kích thước thay đổi)',
        5: 'Khối văn bản dọc (thẳng đứng)',
        6: 'Khối văn bản (nhiều dòng) ← TỐT NHẤT cho tài liệu',
        7: 'Một dòng',
        8: 'Một từ',
        9: 'Một từ dạng circle',
        10: 'Một ký tự',
        11: 'Sparse text (văn bản rải rác)',
        12: 'Sparse text + OSD',
        13: 'Raw line (một dòng thô)',
    }

    def __init__(self, lang='vie', oem=3, psm=6):
        self.lang = lang
        self.oem = oem
        self.psm = psm
        self.config = f'--oem {oem} --psm {psm}'
        # Whitelist: không giới hạn ký tự (để Tesseract nhận hết dấu tiếng Việt)

    def ocr(self, preprocessed_img):
        """Nhận dạng văn bản từ ảnh đã xử lý."""
        pil_img = Image.fromarray(preprocessed_img)
        text = pytesseract.image_to_string(pil_img, lang=self.lang, config=self.config)
        return text.strip()

    def ocr_with_confidence(self, preprocessed_img):
        """Nhận dạng kèm điểm tin cậy (confidence) cho từng từ."""
        pil_img = Image.fromarray(preprocessed_img)
        data = pytesseract.image_to_data(
            pil_img, lang=self.lang, config=self.config,
            output_type=pytesseract.Output.DICT
        )
        results = []
        for i in range(len(data['text'])):
            word = data['text'][i].strip()
            conf = int(data['conf'][i])
            if word and conf > 0:
                results.append({'word': word, 'conf': conf,
                                 'x': data['left'][i], 'y': data['top'][i],
                                 'w': data['width'][i], 'h': data['height'][i]})
        return results

    def visualize_boxes(self, original_img, preprocessed_img, min_conf=50):
        """Vẽ bounding box các từ được nhận dạng lên ảnh gốc."""
        results = self.ocr_with_confidence(preprocessed_img)
        # Scale factor nếu ảnh đã upscale
        orig_h, orig_w = original_img.shape[:2]
        proc_h, proc_w = preprocessed_img.shape[:2]
        sx = orig_w / proc_w
        sy = orig_h / proc_h

        vis = original_img.copy()
        if len(vis.shape) == 2:
            vis = cv2.cvtColor(vis, cv2.COLOR_GRAY2BGR)

        for r in results:
            if r['conf'] >= min_conf:
                x = int(r['x'] * sx)
                y = int(r['y'] * sy)
                w = int(r['w'] * sx)
                h = int(r['h'] * sy)
                color = (0, int(r['conf'] * 2.55), 255 - int(r['conf'] * 2.55))
                cv2.rectangle(vis, (x, y), (x + w, y + h), color, 2)

        return vis

print('✅ Đã định nghĩa VietnameseTesseractOCR!')
print('\nCác chế độ PSM:')
for k, v in VietnameseTesseractOCR.PSM_MODES.items():
    print(f'  PSM {k:>2}: {v}')

## 5. Chạy OCR trên ảnh mẫu

In [ ]:
IMAGE_PATH = '/tmp/sample_viet.png'

# Tiền xử lý
preprocessor = VietnameseOCRPreprocessor()
binary = preprocessor.preprocess(
    IMAGE_PATH,
    binarize_method='sauvola',
    do_upscale=True,
    do_deskew=True,
    do_denoise=True,
    do_contrast=True,
    do_morph=True,
    show=True
)

# OCR
ocr = VietnameseTesseractOCR(lang='vie', oem=3, psm=6)
text = ocr.ocr(binary)

print('━' * 60)
print('KẾT QUẢ OCR TIẾNG VIỆT:')
print('━' * 60)
print(text)
print('━' * 60)

## 6. So sánh các phương pháp Binarization

In [ ]:
METHODS = ['sauvola', 'otsu', 'adaptive']
LABELS = ['Sauvola (tốt nhất)', 'Otsu', 'Adaptive Gaussian']

fig, axes = plt.subplots(2, len(METHODS), figsize=(5 * len(METHODS), 8))

for col, (method, label) in enumerate(zip(METHODS, LABELS)):
    prep = VietnameseOCRPreprocessor()
    binary_m = prep.preprocess(
        IMAGE_PATH, binarize_method=method,
        do_upscale=True, do_deskew=False, do_denoise=True,
        do_contrast=True, do_morph=True, show=False
    )
    text_m = ocr.ocr(binary_m)

    axes[0][col].imshow(binary_m, cmap='gray')
    axes[0][col].set_title(label, fontsize=11)
    axes[0][col].axis('off')

    axes[1][col].text(0.01, 0.99, text_m,
                      transform=axes[1][col].transAxes,
                      fontsize=8, verticalalignment='top',
                      fontfamily='monospace',
                      bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    axes[1][col].axis('off')
    axes[1][col].set_title('Kết quả OCR', fontsize=10)

plt.suptitle('So sánh các phương pháp Binarization', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. So sánh các chế độ PSM

In [ ]:
PSM_TEST = [3, 6, 11]  # Các PSM phổ biến nhất

# Dùng ảnh đã xử lý
print('━' * 70)
print(f'{"PSM":>5}  {"Mô tả":<40}  {"Kết quả (30 ký tự đầu)"}')
print('━' * 70)

for psm in PSM_TEST:
    ocr_psm = VietnameseTesseractOCR(lang='vie', oem=3, psm=psm)
    text_psm = ocr_psm.ocr(binary)
    preview = text_psm.replace('\n', ' ')[:50]
    desc = VietnameseTesseractOCR.PSM_MODES.get(psm, '')[:35]
    print(f'{psm:>5}  {desc:<40}  {preview}')

print('━' * 70)

## 8. Visualize bounding box kết quả nhận dạng

In [ ]:
original = cv2.imread(IMAGE_PATH)
vis_img = ocr.visualize_boxes(original, binary, min_conf=60)

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
plt.title('Ảnh gốc', fontsize=12)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB))
plt.title('Bounding box (màu: xanh = tin cậy cao, đỏ = thấp)', fontsize=12)
plt.axis('off')

plt.suptitle('Kết quả nhận dạng tiếng Việt', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Thống kê confidence
results = ocr.ocr_with_confidence(binary)
confs = [r['conf'] for r in results]
if confs:
    print(f'\nThống kê độ tin cậy (Confidence):')
    print(f'  Trung bình : {np.mean(confs):.1f}%')
    print(f'  Min / Max  : {min(confs)}% / {max(confs)}%')
    print(f'  Số từ nhận dạng: {len(results)}')
    high = sum(1 for c in confs if c >= 80)
    print(f'  Số từ >= 80% tin cậy: {high}/{len(results)}')

## 9. Đánh giá CER / WER (nếu có ground truth)

In [ ]:
from jiwer import wer, cer

GROUND_TRUTH = SAMPLE_TEXT  # Văn bản gốc đã tạo

# Chuẩn hoá: bỏ newline thừa, lowercase
def normalize(text):
    return ' '.join(text.lower().split())

gt_norm = normalize(GROUND_TRUTH)
pred_norm = normalize(text)

cer_score = cer(gt_norm, pred_norm)
wer_score = wer(gt_norm, pred_norm)

print('╔══════════════════════════════════════╗')
print('║      ĐÁNH GIÁ ĐỘ CHÍNH XÁC OCR      ║')
print('╠══════════════════════════════════════╣')
print(f'║  CER (Character Error Rate) : {cer_score:>6.2%}  ║')
print(f'║  WER (Word Error Rate)      : {wer_score:>6.2%}  ║')
print(f'║  Độ chính xác ký tự (1-CER) : {1-cer_score:>6.2%}  ║')
print('╚══════════════════════════════════════╝')

print('\nGround truth (30 ký tự đầu):'), print(gt_norm[:80])
print('\nKết quả OCR (30 ký tự đầu):'  ), print(pred_norm[:80])

## 10. OCR ảnh thực tế (upload ảnh của bạn)

In [ ]:
# ===== HƯỚNG DẪN =====
# 1. Upload ảnh tiếng Việt của bạn lên Google Colab
# 2. Thay đường dẫn bên dưới
# 3. Chạy ô này

YOUR_IMAGE = '/tmp/sample_viet.png'  # ← Thay bằng đường dẫn ảnh của bạn

def run_full_pipeline(image_path, lang='vie', psm=6, binarize='sauvola'):
    """
    Pipeline đầy đủ: đọc ảnh → tiền xử lý → OCR → in kết quả.
    """
    print(f'📷 Đang xử lý: {image_path}')

    # Tiền xử lý
    preprocessor = VietnameseOCRPreprocessor()
    binary = preprocessor.preprocess(
        image_path,
        binarize_method=binarize,
        do_upscale=True,
        do_deskew=True,
        do_denoise=True,
        do_contrast=True,
        do_morph=True,
        show=True
    )

    # OCR
    ocr_engine = VietnameseTesseractOCR(lang=lang, oem=3, psm=psm)
    text = ocr_engine.ocr(binary)

    print('┌─────────────────────────────────────┐')
    print('│         KẾT QUẢ OCR TIẾNG VIỆT      │')
    print('└─────────────────────────────────────┘')
    print(text)
    print('─' * 40)

    # Bounding box
    orig = cv2.imread(image_path)
    vis = ocr_engine.visualize_boxes(orig, binary, min_conf=60)
    plt.figure(figsize=(12, 5))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title('Kết quả nhận dạng (bounding box)', fontsize=13)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    return text

result = run_full_pipeline(YOUR_IMAGE)

## 11. Ghi chú kỹ thuật – Lý do mỗi thuật toán

| Bước | Thuật toán | Lý do chọn |
|---|---|---|
| Upscale | `cv2.INTER_CUBIC` | Dấu tiếng Việt nhỏ, cần ảnh đủ lớn để Tesseract nhận |
| Denoise | Non-local Means / Gaussian | Xóa nhiễu nhưng giữ nét chữ và dấu |
| Deskew | MinAreaRect → WarpAffine | Chỉnh nghiêng giúp Tesseract đọc đúng dòng |
| CLAHE | Contrast Limited AHE | Cân bằng histogram cục bộ, ánh sáng không đều |
| Binarize | **Sauvola** | Thích nghi theo vùng, tốt nhất cho tài liệu có nền không đều |
| Morphology | Close → Open | Lấp nét đứt, loại noise nhỏ (quan trọng với dấu) |
| OCR | `--oem 3 --psm 6` | OEM 3 = LSTM+Legacy, PSM 6 = khối văn bản |
| Lang | `vie` (hoặc `vie+eng`) | Bộ data tiếng Việt đầy đủ dấu |

### Mẹo tăng độ chính xác tiếng Việt:
- **DPI >= 300**: Ảnh chụp/scan cần đủ độ phân giải
- **Sauvola > Otsu** cho ảnh nền không đều
- **CLAHE** giúp nhiều khi ánh sáng thực tế không đều
- **Upscale 2x** giúp Tesseract nhận dấu sắc, huyền, hỏi, ngã, nặng tốt hơn
- **lang='vie+eng'** nếu văn bản trộn Việt-Anh